# Chapitre 12 - Cours 3 : discriminants flexibles (FDA, PDA, MDA)

La seconde moitié du chapitre (sections 12.4 à 12.7) généralise l'analyse discriminante
linéaire (LDA, section 4.3) dans trois directions :

1. **FDA** (*flexible discriminant analysis*) : on réécrit la LDA comme une suite de
   **régressions** (*optimal scoring*), puis on remplace la régression linéaire par une
   régression non paramétrique ;
2. **PDA** (*penalized discriminant analysis*) : on **pénalise** les coefficients quand les
   variables sont nombreuses et corrélées (signaux, images) ;
3. **MDA** (*mixture discriminant analysis*) : chaque classe est un **mélange** de gaussiennes
   de covariance commune, estimé par l'algorithme EM.

Exemples : deux gaussiennes emboîtées (figure 12.9), la simulation « mixture »
(figure 12.13) et les données *waveform* (figures 12.14–12.15, tableau 12.4).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from esl.plotting import set_style, COLORS, BLUE, ORANGE, GRAY, SEQ, DIV, TWO_CLASS
from esl import datasets
from esl.ch12_svm import (LDA, QDA, FDA, MDA, class_indicator, poly_basis, rbf_kernel,
                          second_diff_penalty, simulate_mixture, mixture_bayes_prob,
                          plot_mixture_boundary, waveform_h, simulate_waveform,
                          waveform_bayes_posterior)
set_style()

## 1. Pourquoi généraliser la LDA ?

La LDA est simple et souvent efficace, mais (section 12.4 du livre) :

* ses frontières sont **linéaires**, ce qui est parfois trop rigide ;
* un **seul prototype** (centroïde) par classe peut être insuffisant pour des classes
  hétérogènes ;
* avec **beaucoup de variables corrélées** (pixels, fréquences), elle estime trop de
  paramètres avec une grande variance.

Les trois réponses (FDA, PDA, MDA) reposent sur un même lien entre LDA et régression.

## 2. LDA et *optimal scoring*

Supposons $K$ classes $\mathcal G = \{1,\dots,K\}$. On cherche une fonction de **scores**
$\theta : \mathcal G\to\mathbb R$ telle que les étiquettes transformées $\theta(g_i)$ soient
le mieux possible prédites par une régression linéaire sur $x$ (éq. (12.52)) :

$$\min_{\beta,\theta}\sum_{i=1}^N\left(\theta(g_i) - x_i^T\beta\right)^2,$$

avec des contraintes sur $\theta$ (moyenne nulle, variance unité sur l'échantillon) pour
exclure la solution triviale $\theta = 0$, $\beta = 0$.

Plus généralement, on cherche $L \le K-1$ couples (scores $\theta_\ell$, coefficients
$\beta_\ell$) minimisant le **résidu quadratique moyen** (12.53)

$$\mathrm{ASR} = \frac1N\sum_{\ell=1}^L\sum_{i=1}^N\left(\theta_\ell(g_i) - x_i^T\beta_\ell\right)^2,$$

les scores étant orthonormés pour un produit scalaire convenable.

**Résultat (exercices 12.6 et 12.8).** Les $\beta_\ell$ sont proportionnels aux directions
discriminantes $\nu_\ell$ de la LDA (section 4.3.3), et la distance de Mahalanobis d'un point
$x$ au centroïde $\hat\mu_k$ s'écrit (12.54)–(12.55)

$$\delta_J(x, \hat\mu_k) = \sum_{\ell=1}^{K-1}w_\ell\left(\hat\eta_\ell(x) - \bar\eta_\ell^k\right)^2 + D(x),\qquad w_\ell = \frac{1}{r_\ell^2(1-r_\ell^2)},$$

où $\hat\eta_\ell(x) = x^T\beta_\ell$, $\bar\eta_\ell^k$ est sa moyenne dans la classe $k$,
$r_\ell^2$ le résidu quadratique moyen du $\ell$-ième ajustement, et $D(x)$ ne dépend pas de $k$.

> **À retenir.** La LDA = une suite de régressions linéaires + un classement au centroïde le
> plus proche dans l'espace des valeurs ajustées.